<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 从零实现用于 LLM 对齐的直接偏好优化（DPO）

- 本 notebook 从零实现直接偏好优化（Direct Preference Optimization，DPO），并将其应用于大语言模型（LLM），以提升模型生成更符合用户偏好的回复的能力

In [ ]:
# !pip install -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/requirements.txt

In [ ]:
from importlib.metadata import version

pkgs = [
    "tiktoken",    # 分词器
    "torch",       # 深度学习库
]
for p in pkgs:
    print(f"{p} 版本: {version(p)}")

&nbsp;
# 1) DPO 简介

- DPO 在论文 [Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290) 中提出，是用于微调大语言模型（LLM）的人类反馈强化学习（RLHF）的一种替代方案
- DPO 可用于微调（或对齐）模型，使其生成更符合用户期望与指令的回复

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/dpo/1.webp" width=500px>

- 在指令微调中，我们训练 LLM 在给定提示下生成正确答案
- 然而实践中，给出正确答案的方式有多种，且正确答案在风格上可能不同；例如，当要求 LLM 给出购买笔记本电脑的建议时，可能出现技术性回复与更面向用户的回复，如下图所示

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/dpo/2.webp" width=700px>

- RLHF 和 DPO 可用于教 LLM 偏好某种回答风格而非另一种，即更好地与用户偏好对齐
- 需要训练单独奖励模型的 RLHF 流程如下所示

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/dpo/4.webp" width=600px>

- 与 RLHF 相比，DPO 旨在简化流程：直接针对用户偏好优化模型，无需复杂的奖励建模与策略优化
- 换言之，DPO 专注于直接优化模型输出，使其与人类偏好或特定目标对齐
- 下图概述了 DPO 的核心思想

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/dpo/5.webp?123" width=600px>

- 实现 DPO 损失的具体公式如下；我们将在本 notebook 后续用 Python 实现时再回顾该公式

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/dpo/3.webp?123" width=600px>

- 在上式中，
  - "expected value"（期望值）$\mathbb{E}$ 是统计学术语，表示随机变量（括号内表达式）的平均值或均值；优化 $-\mathbb{E}$ 可使模型更好地与用户偏好对齐
  - 变量 $\pi_{\theta}$ 即所谓的 policy（策略，借自强化学习术语），表示我们要优化的 LLM；$\pi_{ref}$ 是 reference LLM（参考 LLM），通常是优化前的原始 LLM（训练开始时，$\pi_{\theta}$ 与 $\pi_{ref}$ 通常相同）
  - $\beta$ 是控制 $\pi_{\theta}$ 与参考模型之间散度的超参数；增大 $\beta$ 会减小 $\pi_{\theta}$ 与 $\pi_{ref}$ 在对数概率差异对整体损失函数影响上的权重，从而降低两模型之间的散度
  - logistic sigmoid 函数 $\sigma(\centerdot)$ 将 preferred 与 rejected 回复的对数几率（logistic sigmoid 函数内的项）转换为概率分数
- 为避免在本 notebook 中展开过多细节，我未来可能会单独撰写更详细的文章
- 与此同时，若您有兴趣比较 RLHF 与 DPO，请参阅我的文章 [Tips for LLM Pretraining and Evaluating Reward Models](https://magazine.sebastianraschka.com/p/tips-for-llm-pretraining-and-evaluating-rms) 中的 [2.2. RLHF vs Direct Preference Optimization (DPO)](https://magazine.sebastianraschka.com/i/142924793/rlhf-vs-direct-preference-optimization-dpo) 一节

&nbsp;
# 2) 为 DPO 准备偏好数据集

- 让我们从加载并准备数据集开始；在回顾 DPO 损失公式之前，这可能已解答您许多疑问
- 此处我们使用包含对指令提示更礼貌与较不礼貌回复的数据集（下一节有具体示例）
- 该数据集通过 [create-preference-data-ollama_ch.ipynb](create-preference-data-ollama_ch.ipynb) notebook 生成

&nbsp;
## 2.1) 加载偏好数据集

- 数据集为包含 1100 条记录的 json 文件：

In [ ]:
import json
import os
import requests


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    data = json.loads(text_data)
    return data


file_path = "instruction-data-with-preference.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/04_preference-tuning-with-dpo/instruction-data-with-preference.json"
)

data = download_and_load_file(file_path, url)
print("条目数:", len(data))

- 让我们查看两条示例记录：

In [ ]:
import pprint

pprint.pp(data[50])

In [ ]:
pprint.pp(data[999])



```
# This is formatted as code
```

- 如上所示，数据集包含 5 个键：
    - `'instruction'` 与 `'input'` 用作 LLM 输入
    - `'output'` 包含模型在第 7 章指令微调步骤中训练所用的回复
    - `'chosen'` 与 `'rejected'` 是 DPO 使用的条目；此处 `'chosen'` 为 preferred 回复，`'rejected'` 为 dispreferred 回复
- 目标是让模型遵循 chosen 而非 rejected 回复的风格

- 下面是按 Alpaca 提示风格格式化模型输入的工具函数，与第 7 章类似（[../01_main-chapter-code/ch07_ch.ipynb](../01_main-chapter-code/ch07_ch.ipynb)）：

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [ ]:
model_input = format_input(data[50])
print(model_input)

- 类似地，我们可按 Alpaca 提示风格格式化 chosen 与 rejected 回复：

In [ ]:
desired_response = f"### Response:\n{data[50]['chosen']}"
print(desired_response)

In [ ]:
possible_response = f"### Response:\n{data[50]['rejected']}"
print(possible_response)

&nbsp;
## 2.2) 创建训练、验证与测试划分

- 接下来，我们将数据集划分为 3 个子集：85% 训练、5% 验证、10% 测试：

In [ ]:
train_portion = int(len(data) * 0.85)  # 85% 用于训练
test_portion = int(len(data) * 0.1)    # 10% 用于测试
val_portion = len(data) - train_portion - test_portion  # 剩余 5% 用于验证

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [ ]:
print("训练集长度:", len(train_data))
print("验证集长度:", len(val_data))
print("测试集长度:", len(test_data))

&nbsp;
## 2.3) 实现 `PreferenceDataset` 类与批处理函数

- 本节我们改写第 7 章的 `InstructionDataset` 类（[../01_main-chapter-code/ch07_ch.ipynb](../01_main-chapter-code/ch07_ch.ipynb)）以用于 DPO
- 这意味着不再只关注单一输出序列（回复），而是修改数据集类以返回一对回复，其中一条 preferred（"chosen"）于另一条（"rejected"）
- 总体而言，`PreferenceDataset` 与第 7 章使用的 `InstructionDataset` 几乎相同：

In [ ]:
import torch
from torch.utils.data import Dataset


class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # 预分词文本
        self.encoded_texts = []
        for entry in data:
            prompt = format_input(entry)
            rejected_response = entry["rejected"]
            chosen_response = entry["chosen"]

            prompt_tokens = tokenizer.encode(prompt)
            chosen_full_text = f"{prompt}\n\n### Response:\n{chosen_response}"
            rejected_full_text = f"{prompt}\n\n### Response:\n{rejected_response}"
            chosen_full_tokens = tokenizer.encode(chosen_full_text)
            rejected_full_tokens = tokenizer.encode(rejected_full_text)

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


- 除更新的 `PreferenceDataset` 类外，我们还需要更新的 batch collation 函数，用于将各 batch 中序列填充到相同长度以便组 batch
- 我在下方代码中添加了注释以说明流程；不过，通过下方示例输入输出可能最容易理解其工作原理：

In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    allowed_max_length=None,
    mask_prompt_tokens=True,
    device="cpu"
):
    # 初始化列表以保存 batch 数据
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": []

    }

    # 确定最长序列以设置统一 padding 长度
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key])+1 for item in batch)
            max_length_common = max(max_length_common, current_max)

    # 处理 batch 中每条样本
    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            # 按统一最大长度调整 padding
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            mask = torch.ones(len(padded)).bool()

            # 将所有 padding token 的 mask 设为 False
            mask[len(sequence):] = False

            # 将所有输入 token 的 mask 设为 False
            # +2 将 "### Response" 前的 2 个换行 ("\n") token 设为 False
            if mask_prompt_tokens:
                mask[:prompt.shape[0]+2] = False

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    # 最终处理
    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        # 将给定键的所有序列堆叠为 tensor
        tensor_stack = torch.stack(batch_data[key])

        # 可选地截断到最大序列长度
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]

        # 移动到指定 device
        batch_data[key] = tensor_stack.to(device)

    return batch_data

- 在使用自定义 collate 函数之前，先用部分函数参数预填充创建一个版本：

In [ ]:
from functools import partial

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # 使用 PyTorch 2.9 或更新版本以获得稳定的 mps 结果
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("设备:", device)

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,            # 若可用，将数据直接放到 GPU
    mask_prompt_tokens=True,  # 可选
    allowed_max_length=1024   # 模型支持的上下文长度
)

- 现在让我们看看 `customized_collate_fn` 的实际效果，将其应用于偏好数据集的前两条样本：

In [ ]:
example_data = data[:2]

for i in example_data:
    print()
    pprint.pp(i)

- 接下来，实例化 `example_dataset`，并用 PyTorch `DataLoader` 创建 `example_dataloader`，模拟后续模型训练将使用的 data loader：

In [ ]:
import tiktoken
from torch.utils.data import DataLoader


tokenizer = tiktoken.get_encoding("gpt2")

example_dataset = PreferenceDataset(example_data, tokenizer)

example_dataloader = DataLoader(
    example_dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False
)

- 数据集包含以下键：

In [ ]:
for batch in example_dataloader:
    break

print("batch.keys:", batch.keys())

- prompt 是 tensor 列表，每个 tensor 包含某条样本的 token ID；由于 batch size 为 2，此处有两个 token ID tensor 列表：

In [ ]:
batch["prompt"]

- 训练时我们并不真正需要 prompt 文本；训练期间需要喂给模型的是 `"chosen"` 与 `"rejected"` 条目
- `"chosen"` 与 `"rejected"` 回复条目经填充以便堆叠为 tensor；与 prompt 类似，这些回复文本被编码为 token ID：

In [ ]:
batch["chosen"]

- 上述 token ID 代表模型输入，但在此格式下人类难以解读
- 因此，我们实现一个小工具函数将其转回文本，便于检查与理解：

In [ ]:
def decode_tokens_from_batch(token_ids, tokenizer):
    ids_in_python_list = token_ids.flatten().tolist()
    return tokenizer.decode(ids_in_python_list)

- 将 `decode_tokens_from_batch` 工具函数应用于 batch 中第一条 prompt：

In [ ]:
text = decode_tokens_from_batch(
    token_ids=batch["prompt"][0],  # [0] 表示 batch 中第一条
    tokenizer=tokenizer,
)
print(text)

- 如上所示，prompt 已正确格式化；现在对 `"chosen"` 回复做同样操作：

In [ ]:
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0],
    tokenizer=tokenizer,
)
print(text)

- 如上所示，与指令微调类似，训练期间传给模型的回复也包含输入 prompt
- 另请注意，我们使用 `<|endoftext|>` token 作为 padding token，以便将回复扩展到相近长度以组 batch
- 请放心；`<|endoftext|>` token 在后续计算损失时会被忽略，不会影响训练结果
- 现在让我们也检查对应的 rejected 回复：

In [ ]:
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0],
    tokenizer=tokenizer,
)
print(text)

- 如上所示，此例中 rejected 回复是 chosen 回复的较不礼貌版本（我们不希望模型生成不礼貌回复）
- 最后谈谈 data mask：若仔细查看上方实现的 custom collate 函数，我们为每条数据集记录创建了 `"chosen_mask"` 与 `"rejected_mask"`
- mask 与 `"chosen"` 条目形状相同，如下所示：

In [ ]:
print("chosen 输入:", batch["chosen"][0].shape)
print("chosen mask:  ", batch["chosen_mask"][0].shape)

- 这些 mask 的内容为布尔值（`True` 与 `False`）：

In [ ]:
batch["chosen_mask"][0]

- `True` 值表示对应实际回复的 token ID
- `False` token 对应 prompt token（若在 `customized_collate_fn` 中设置 `mask_prompt_tokens=True`，我们此前已设置）或 padding token
- 因此，可用 mask 作为选择 mask 仅选取对应回复的 token ID，即去掉所有 prompt 与 padding token，如下所示：

In [ ]:
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0][batch["chosen_mask"][0]],
    tokenizer=tokenizer,
)
print(text)

In [ ]:
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0][batch["rejected_mask"][0]],
    tokenizer=tokenizer,
)
print(text)

- 后续计算 DPO 损失时，我们将用此 mask 忽略 prompt 与 padding token

&nbsp;
## 2.4 创建训练、验证与测试集数据加载器

- 上文我们使用偏好数据集的小示例子集进行说明
- 现在创建实际的训练、验证与测试集数据加载器
- 该过程与预训练及指令微调章节中创建数据加载器相同，应不言自明

In [ ]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = PreferenceDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [ ]:
val_dataset = PreferenceDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = PreferenceDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

- 让我们遍历数据加载器并查看数据集形状：

In [ ]:
print("训练 loader:")
for batch in train_loader:
    print(
        batch["chosen"].shape,
        batch["rejected"].shape,
    )

- 每行显示该 batch 中 `"chosen"` 与 `"rejected"` 条目的形状
- 由于我们按 batch 进行填充，每行形状不同
- 这是出于效率考虑，若将整个数据集所有样本填充到最长样本长度将非常低效

&nbsp;
# 3) 加载用于 DPO 对齐的微调 LLM

- LLM 对齐步骤（如 RLHF 或 DPO）假定我们已有指令微调模型
- 本节含最少代码，用于加载第 7 章指令微调并保存的模型（通过 [../01_main-chapter-code/ch07_ch.ipynb](../01_main-chapter-code/ch07_ch.ipynb)）
- 请先运行第 7 章代码创建指令微调模型再继续
- 下方代码将把指令微调模型复制到当前目录：

In [ ]:
from pathlib import Path
import shutil


finetuned_model_path = Path("gpt2-medium355M-sft.pth")
if not finetuned_model_path.exists():

    # 尝试在本地查找模型 checkpoint：
    relative_path = Path("..") / "01_main-chapter-code" / finetuned_model_path
    if relative_path.exists():
        shutil.copy(relative_path, ".")

    # 若在 Google Colab 上运行本 notebook，从 Google Drive 文件夹获取
    elif "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        google_drive_path = "/content/drive/My Drive/Books/LLMs-From-Scratch/ch07/colab/gpt2-medium355M-sft.pth"  # 读者需调整此路径
        shutil.copy(google_drive_path, ".")

    else:
        print(
            f"找不到 '{finetuned_model_path}'.\n"
            "请运行 `ch07_ch.ipynb` notebook 以微调并保存微调后的模型。"
        )

- 接下来，我们复用前几章的基本配置加载模型权重：

In [ ]:
from previous_chapters import GPTModel
# 若本地没有 `previous_chapters.py` 文件，
# 可从 `llms-from-scratch` PyPI 包导入。
# 详情见：https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 例如：
# from llms_from_scratch.ch04 import GPTModel


BASE_CONFIG = {
    "vocab_size": 50257,     # 词汇表大小
    "context_length": 1024,  # 上下文长度
    "drop_rate": 0.0,        # Dropout 率
    "qkv_bias": True         # Query-key-value 偏置
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model = GPTModel(BASE_CONFIG)

In [ ]:
model.load_state_dict(
    torch.load(
        "gpt2-medium355M-sft.pth",
        map_location=torch.device("cpu"),
        weights_only=True
    )
)
model.eval();

- 在用 DPO 训练加载的模型之前，让我们用部分样本测试，确保微调模型已正确保存与加载：

In [ ]:
prompt = """Below is an instruction that describes a task. Write a response
that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'
"""

In [ ]:
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)
# 或者：
# from llms_from_scratch.ch05 (
#     generate,
#     text_to_token_ids,
#     token_ids_to_text
# )

torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(prompt, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

response = token_ids_to_text(token_ids, tokenizer)
print(response)

- 如上所示，模型给出了合理且正确的回复
- 如第 7 章所述，实践中我们会清理回复，仅返回去掉 prompt 与 prompt 风格的回复文本（类似 ChatGPT 等您熟悉的体验）：

In [ ]:
def extract_response(response_text, input_text):
    return response_text[len(input_text):].replace("### Response:", "").strip()

response = extract_response(response, prompt)
print(response)

- 现在，我们几乎可以进入 DPO 部分
- 如本 notebook 开头所述，DPO 使用两个 LLM：policy model（要优化的 LLM）与 reference model（保持不变的原始模型）
- 下面，我们将 `model` 重命名为 `policy_model`，并实例化第二个模型作为 `reference_model`

In [ ]:
policy_model = model

reference_model = GPTModel(BASE_CONFIG)
reference_model.load_state_dict(
    torch.load(
        "gpt2-medium355M-sft.pth",
        map_location=torch.device("cpu"),
        weights_only=True
    )
)
reference_model.eval()

policy_model.to(device)
reference_model.to(device);

&nbsp;
# 4) 编写 DPO 损失函数

- 在前几节完成模型加载与数据集准备后，现在进入核心部分：编写 DPO 损失
- 请注意，下方 DPO 损失代码基于 [Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290) 论文中的方法
- 为便于参考，核心 DPO 公式再次展示如下：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/dpo/3.webp?123" width=800px>

- 在上式中，
  - "expected value"（期望值）$\mathbb{E}$ 是统计学术语，表示随机变量（括号内表达式）的平均值或均值；优化 $-\mathbb{E}$ 可使模型更好地与用户偏好对齐
  - 变量 $\pi_{\theta}$ 即所谓的 policy（策略），表示我们要优化的 LLM；$\pi_{ref}$ 是 reference LLM，通常是优化前的原始 LLM（训练开始时，$\pi_{\theta}$ 与 $\pi_{ref}$ 通常相同）
  - $\beta$ 是控制 $\pi_{\theta}$ 与参考模型之间散度的超参数；增大 $\beta$ 会增大 $\pi_{\theta}$ 与 $\pi_{ref}$ 在对数概率差异对整体损失函数影响上的权重，从而增大两模型之间的散度
  - logistic sigmoid 函数 $\sigma(\centerdot)$ 将 preferred 与 rejected 回复的对数几率转换为概率分数
- 在代码中，可按如下方式实现 DPO 损失：

In [ ]:
import torch.nn.functional as F

def compute_dpo_loss(
      model_chosen_logprobs,
      model_rejected_logprobs,
      reference_chosen_logprobs,
      reference_rejected_logprobs,
      beta=0.1,
    ):
    """计算一批 policy 与 reference model 对数概率的 DPO 损失。

    Args:
        model_chosen_logprobs: policy model 对 chosen 回复的对数概率。形状：(batch_size,)
        model_rejected_logprobs: policy model 对 rejected 回复的对数概率。形状：(batch_size,)
        reference_chosen_logprobs: reference model 对 chosen 回复的对数概率。形状：(batch_size,)
        reference_rejected_logprobs: reference model 对 rejected 回复的对数概率。形状：(batch_size,)
        beta: DPO 损失的温度参数；通常在 0.1 到 0.5 之间。当 beta -> 0 时我们忽略 reference model。

    Returns:
        三个 tensor 的元组：(loss, chosen_rewards, rejected_rewards)。
    """

    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs
    logits = model_logratios - reference_logratios

    # DPO (Eq. 7 of https://arxiv.org/pdf/2305.18290.pdf)
    losses = -F.logsigmoid(beta * logits)

    # 可选值，用于跟踪训练进度
    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    # .mean() 对 batch 中样本取平均
    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()

- 若您熟悉对数，请注意一般关系 $\log\left(\frac{a}{b}\right) = \log a - \log b$，我们已在上方代码中应用
- 牢记这一点，让我们逐步说明（`logprobs` 将用单独函数计算）
- 首先从以下两行开始

    ```python
    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs
    ```

- 上述行计算 policy model 与 reference model 对 chosen 与 rejected 样本的对数概率（logits）差（这源于 $\log\left(\frac{a}{b}\right) = \log a - \log b$）：

$$\log \left( \frac{\pi_\theta (y_w \mid x)}{\pi_\theta (y_l \mid x)} \right) \quad \text{and} \quad \log \left( \frac{\pi_{\text{ref}}(y_w \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right)$$

- 接下来，代码 `logits = model_logratios - reference_logratios` 计算模型 log ratio 与 reference model log ratio 之差，即

$$\beta \log \left( \frac{\pi_\theta (y_w \mid x)}{\pi_{\text{ref}} (y_w \mid x)} \right)
- \beta \log \left( \frac{\pi_\theta (y_l \mid x)}{\pi_{\text{ref}} (y_l \mid x)} \right)$$


- 最后，`losses = -F.logsigmoid(beta * logits)` 使用 log-sigmoid 函数计算损失；在原公式中，期望内的项为

$$\log \sigma \left( \beta \log \left( \frac{\pi_\theta (y_w \mid x)}{\pi_{\text{ref}} (y_w \mid x)} \right)
- \beta \log \left( \frac{\pi_\theta (y_l \mid x)}{\pi_{\text{ref}} (y_l \mid x)} \right) \right)$$

- 上文我们假定对数概率已计算；现在定义 `compute_logprobs` 函数，用于计算传入 `compute_dpo_loss` 的对数概率，即 $\pi_\theta (y_w \mid x)$、${\pi_\theta (y_l \mid x)}$ 等：

In [ ]:
def compute_logprobs(logits, labels, selection_mask=None):
    """
    计算对数概率。

    Args:
      logits: 形状为 (batch_size, num_tokens, vocab_size) 的 Tensor
      labels: 形状为 (batch_size, num_tokens) 的 Tensor
      selection_mask: 形状为 (batch_size, num_tokens) 的 Tensor

    Returns:
      mean_log_prob: 排除 padding token 后的平均对数概率。
    """

    # labels 为输入右移一位
    labels = labels[:, 1:].clone()

    # 截断 logits 以匹配 labels 的 token 数
    logits = logits[:, :-1, :]

    log_probs = F.log_softmax(logits, dim=-1)

    # 收集实际 labels 的对数概率
    selected_log_probs = torch.gather(
        input=log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()

        # 应用 mask 过滤 padding token
        selected_log_probs = selected_log_probs * mask

        # 计算排除 padding token 后的平均对数概率
        # 对 token 取平均，因此形状为 (batch_size,)
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)

        return avg_log_prob

    else:
        return selected_log_probs.mean(-1)

- 请注意，上述函数因 `torch.gather` 可能初看有些吓人，但其与 PyTorch `cross_entropy` 函数底层所做之事颇为相似
- 例如，考虑以下示例：

In [ ]:
# 示例数据
logits = torch.tensor(
    [[2.0, 1.0, 0.1],
     [0.5, 2.5, 0.3]])  # 形状：(2, 3)
targets = torch.tensor([0, 2])  # 形状：(2,)


# 使用 torch.gather 手动计算损失
log_softmax_logits = F.log_softmax(logits, dim=1)  # 形状：(2, 3)
selected_log_probs = torch.gather(
    input=log_softmax_logits,
    dim=1,
    index=targets.unsqueeze(1), # 形状 2, 1
).squeeze(1)  # 形状：(2,)
manual_loss = -selected_log_probs.mean()  # 对 batch 取平均


# PyTorch 损失
cross_entropy_loss = F.cross_entropy(logits, targets)

print(manual_loss, cross_entropy_loss)

- 如上所示，两种实现等价；让我们进一步聚焦 `torch.gather` 机制
- 考虑以下两个 tensor：

In [ ]:
t = torch.tensor(
  [[1., 2.,],
   [3., 4.]]
)

m = torch.tensor(
  [[1, 1],
   [0, 1]]
)

- 上式中，`t` 是要从中选取的 tensor，`m` 是指定如何选取的 mask
 - 例如，由于 `m` 第一行包含 `[1, 1]`，它将两次选取 `t` 在索引位置 `1` 的值，即 2。
 - `m` 的第二行 `[0, 1]` 选取 `t` 第二行的索引位置 0 与 1，即 `3.` 与 `4.`

In [ ]:
torch.gather(input=t, dim=-1, index=m)

- 换言之，`torch.gather` 是选择函数
- 早前计算损失时，我们用它检索 50,257 token 词表中对应正确 token 的对数概率
- "正确" token 即 response 条目中的 token

- 关于上述 `compute_logprobs` 函数，我们使用 `torch.gather` 是因为它比 `cross_entropy` 提供更多控制，但本质上是类似思路
- 其中的 `selection_mask` 用于可选地忽略 prompt 与 padding token
- 然后可按如下方式使用 `compute_logprobs` 计算 `compute_dpo_loss` 的输入

In [ ]:
def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):
    """计算输入 batch 的 DPO 损失"""

    # 其中 policy_model(batch["chosen"]) 为 logits
    policy_chosen_log_probas = compute_logprobs(
        logits=policy_model(batch["chosen"]),
        labels=batch["chosen"],
        selection_mask=batch["chosen_mask"]
    )
    policy_rejected_log_probas = compute_logprobs(
        logits=policy_model(batch["rejected"]),
        labels=batch["rejected"],
        selection_mask=batch["rejected_mask"]
    )
    
    with torch.no_grad():
        ref_chosen_log_probas = compute_logprobs(
            logits=reference_model(batch["chosen"]),
            labels=batch["chosen"],
            selection_mask=batch["chosen_mask"]
        )
        ref_rejected_log_probas = compute_logprobs(
            logits=reference_model(batch["rejected"]),
            labels=batch["rejected"],
            selection_mask=batch["rejected_mask"]
        )
    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_logprobs=policy_chosen_log_probas,
        model_rejected_logprobs=policy_rejected_log_probas,
        reference_chosen_logprobs=ref_chosen_log_probas,
        reference_rejected_logprobs=ref_rejected_log_probas,
        beta=beta
    )
    return loss, chosen_rewards, rejected_rewards

- 上述函数适用于单个 batch，例如：

In [ ]:
with torch.no_grad():
    loss = compute_dpo_loss_batch(batch, policy_model, reference_model, beta=0.1)
print(loss)

- 下面，我们将此函数扩展为对 data loader 中指定 `num_batches` 生效：

In [ ]:
def compute_dpo_loss_loader(data_loader, policy_model, reference_model, beta, num_batches=None):
    """将 compute_dpo_loss_batch 应用于整个 data loader"""

    total_loss, total_chosen_rewards, total_rejected_rewards = 0., 0., 0.
    if len(data_loader) == 0:
        return float("nan")

    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # 减少 batch 数以匹配 data loader 中的 batch 总数
        # 若 num_batches 超过 data loader 中的 batch 数
        num_batches = min(num_batches, len(data_loader))
    for i, batch in enumerate(data_loader):
        if i < num_batches:
            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch=batch,
                policy_model=policy_model,
                reference_model=reference_model,
                beta=beta
            )
            total_loss += loss.item()
            total_chosen_rewards += chosen_rewards.item()
            total_rejected_rewards += rejected_rewards.item()

        else:
            break

    # 计算平均值
    total_loss /= num_batches
    total_chosen_rewards /= num_batches
    total_rejected_rewards /= num_batches
    return total_loss, total_chosen_rewards, total_rejected_rewards

- 为何指定 `num_batches`？纯粹出于效率（每次在整个数据集上计算损失会显著拖慢训练）

- 最后，为后续训练函数定义便捷函数；`evaluate_dpo_loss_loader` 为训练与验证 loader 计算 DPO 损失与 reward，用于日志记录：

In [ ]:
def evaluate_dpo_loss_loader(policy_model, reference_model, train_loader, val_loader, beta, eval_iter):
    """计算训练与验证数据集的 DPO 损失"""

    policy_model.eval()
    with torch.no_grad():
        train_loss, train_chosen_rewards, train_rejected_rewards = compute_dpo_loss_loader(
            data_loader=train_loader,
            policy_model=policy_model,
            reference_model=reference_model,
            beta=beta,
            num_batches=eval_iter
        )

        val_loss, val_chosen_rewards, val_rejected_rewards = compute_dpo_loss_loader(
            data_loader=val_loader,
            policy_model=policy_model,
            reference_model=reference_model,
            beta=beta,
            num_batches=eval_iter
        )

    res = {
        "train_loss": train_loss,
        "train_chosen_reward": train_chosen_rewards,
        "train_rejected_reward": train_rejected_rewards,
        "val_loss": val_loss,
        "val_chosen_reward": val_chosen_rewards,
        "val_rejected_reward": val_rejected_rewards
    }

    policy_model.train()
    return res

- 本节我们涵盖了大量内容，简要回顾：
  - 流程为：通过模型计算 `logits` $\rightarrow$ 从 logits 计算 `compute_logprobs` $\rightarrow$ 从对数概率计算 `compute_dpo_loss`
  - 我们有 `compute_dpo_loss_batch` 函数促进上述流程
  - `compute_dpo_loss_loader` 工具函数将 `compute_dpo_loss_batch` 应用于 data loader
  - `evaluate_dpo_loss_loader` 将 `compute_dpo_loss_batch` 应用于训练与验证集 data loader 以记录日志

&nbsp;
# 5) 训练模型

- 在前一节设置 DPO 损失函数后，现在终于可以训练模型
- 请注意，该训练函数与预训练及指令微调所用相同，仅有细微差异：
 - 将 cross-entropy 损失替换为新的 DPO 损失函数
 - 还跟踪 reward 与 reward margin，这在 RLHF 与 DPO 中常用于跟踪训练进度

- 开始训练前，让我们打印初始损失与 reward：

In [ ]:
from previous_chapters import generate_and_print_sample
# 或者：
# from llms_from_scratch.ch04 import generate_text_simple


def train_model_dpo_simple(
    policy_model, reference_model, train_loader, val_loader,
    optimizer, num_epochs, beta,
    eval_freq, eval_iter, start_context, tokenizer
):

    # 初始化列表以跟踪损失与已见 token
    tracking = {
        "train_losses": [],
        "train_chosen_rewards": [],
        "train_rejected_rewards": [],
        "val_losses": [],
        "val_chosen_rewards": [],
        "val_rejected_rewards": [],
        "tokens_seen": []
    }
    tokens_seen, global_step = 0, -1

    # 主训练循环
    for epoch in range(num_epochs):
        policy_model.train()  # 将模型设为训练模式

        for batch in train_loader:

            optimizer.zero_grad()  # 重置上一 batch 迭代的损失梯度

            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch=batch,
                policy_model=policy_model,
                reference_model=reference_model,
                beta=beta
            )

            loss.backward()  # 计算损失梯度
            optimizer.step()  # 用损失梯度更新模型权重

            tokens_seen += batch["chosen"].numel()
            global_step += 1

            # 可选评估步骤
            if global_step % eval_freq == 0:
                res = evaluate_dpo_loss_loader(
                    policy_model=policy_model,
                    reference_model=reference_model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    beta=beta,
                    eval_iter=eval_iter
                )
                tracking["train_losses"].append(res["train_loss"])
                tracking["train_chosen_rewards"].append(res["train_chosen_reward"])
                tracking["train_rejected_rewards"].append(res["train_rejected_reward"])
                tracking["val_losses"].append(res["val_loss"])
                tracking["val_chosen_rewards"].append(res["val_chosen_reward"])
                tracking["val_rejected_rewards"].append(res["val_rejected_reward"])
                tracking["tokens_seen"].append(tokens_seen)
                train_reward_margin = res["train_chosen_reward"] - res["train_rejected_reward"]
                val_reward_margin = res["val_chosen_reward"] - res["val_rejected_reward"]

                print(
                    f"Epoch {epoch+1}（Step {global_step:06d}）："
                    f"训练损失 {res['train_loss']:.3f}，验证损失 {res['val_loss']:.3f}，"
                    f"训练 reward margin {train_reward_margin:.3f}，"
                    f"验证 reward margin {val_reward_margin:.3f}"
                )

        # 每个 epoch 后打印样本文本
        generate_and_print_sample(
            model=model,
            tokenizer=tokenizer,
            device=loss.device,
            start_context=start_context
        )

    return tracking

In [ ]:
torch.manual_seed(123) # 因 data loader 中的 shuffle，为可复现性设置种子

res = evaluate_dpo_loss_loader(
    policy_model=policy_model,
    reference_model=reference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    beta=0.1,
    eval_iter=5
)

print("训练损失:", res["train_loss"])
print("验证损失:", res["val_loss"])

print("训练 reward margin:", res["train_chosen_reward"] - res["train_rejected_reward"])
print("验证 reward margin:", res["val_chosen_reward"] - res["val_rejected_reward"])

- 此外，让我们查看部分初始模型回复（验证集前 3 条示例）：

In [ ]:
torch.manual_seed(123)


for entry in val_data[:3]:

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
)

    print(input_text)
    print(f"\n正确回复:\n>> {entry['output']}")
    print(f"\n模型回复:\n>> {response_text.strip()}")
    print("\n-------------------------------------\n")

- 如上所示为原始模型回复
- 请注意，DPO 的目标是诱导轻微的风格变化；即我们希望模型生成相似但稍更礼貌的回复
- 执行下方开始训练的代码单元前，关于部分设置的说明：
 - 我们仅将 policy model 的参数传入 `AdamW` 优化器；那是我们要优化的模型（不修改 reference model）
 - 仅训练 1 个 epoch；因为 DPO 极易 collapse（损失可能改善，但模型开始生成无意义文本）
 - DPO 中最好使用非常小的学习率
 - beta 值可从 0.1 增至 0.5 以减弱 DPO 效果（此处用 0.1 使结果更明显）
 - 在 A100 GPU 上训练约 2 分钟，较小 L4 GPU 约 4 分钟；M3 MacBook Air 约 30 分钟

In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)


optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-6, weight_decay=0.01)

num_epochs = 1
tracking = train_model_dpo_simple(
    policy_model=policy_model,
    reference_model=reference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_epochs=num_epochs,
    beta=0.1, # 取值在 0.1 到 0.5 之间
    eval_freq=5,
    eval_iter=5,
    start_context=format_input(val_data[2]),
    tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"训练完成，耗时 {execution_time_minutes:.2f} 分钟。")

- 根据上方跟踪结果，损失在改善
- 此外，reward margin（chosen 与 rejected 回复 reward 之差）也在改善，这是好迹象
- 下一节我们将更具体地查看这些结果

&nbsp;
# 6) 分析结果

- 让我们通过绘制 DPO 损失开始分析结果：

In [ ]:
from previous_chapters import plot_losses
# 或者：
# from llms_from_scratch.ch05 import plot_losses


epochs_tensor = torch.linspace(0, num_epochs, len(tracking["train_losses"]))
plot_losses(
    epochs_seen=epochs_tensor,
    tokens_seen=tracking["tokens_seen"],
    train_losses=tracking["train_losses"],
    val_losses=tracking["val_losses"],
    label="损失"
)

- 如上所示，损失持续改善，这是好迹象
- 根据下降趋势，您可能想再训练一会儿（读者可尝试），但请注意 DPO 易 collapse，模型可能开始生成无意义回复
- 接下来，让我们查看 reward margin：

In [ ]:
train_reward_margins = [i-j for i,j in zip(tracking["train_chosen_rewards"], tracking["train_rejected_rewards"])]
val_reward_margins = [i-j for i,j in zip(tracking["val_chosen_rewards"], tracking["val_rejected_rewards"])]

plot_losses(
    epochs_seen=epochs_tensor,
    tokens_seen=tracking["tokens_seen"],
    train_losses=train_reward_margins,
    val_losses=val_reward_margins,
    label="奖励边际"
)

- 如上所示，且符合预期，reward margin 在改善；这与损失曲线一致，是好迹象
- 请注意，DPO 损失与 reward margin 是训练期间有价值指标；但它们不能说明全部
- 最后也是最重要的，我们必须对回复进行定性检查
- 此处我们将查看回复（此外，您可用 LLM 对回复评分，类似第 7 章）

In [ ]:
torch.manual_seed(123)


for entry in val_data[:3]:

    input_text = format_input(entry)

    token_ids = generate(
        model=reference_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    reference_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    token_ids = generate(
        model=policy_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    policy_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\n正确回复:\n>> {entry['output']}")
    print(f"\n参考模型回复:\n>> {reference_response_text.strip()}")
    print(f"\n策略模型回复:\n>> {policy_response_text.strip()}")
    print("\n-------------------------------------\n")

- 根据上方 reference model 与 policy model 回复，优化后的模型（即 policy model）相较原始模型（即 reference model）确实略微改变了风格
- 例如，`"Dance" can be classified as a verb.` 变为 `The input string "Dance" could be classified as a verb.`，这是稍更礼貌的回复（使用 "could" 而非 "can" 使陈述听起来不那么断言、更委婉）

In [ ]:
torch.manual_seed(123)


for entry in test_data[:3]:

    input_text = format_input(entry)

    token_ids = generate(
        model=reference_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    reference_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    token_ids = generate(
        model=policy_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    policy_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\n正确回复:\n>> {entry['output']}")
    print(f"\n参考模型回复:\n>> {reference_response_text.strip()}")
    print(f"\n策略模型回复:\n>> {policy_response_text.strip()}")
    print("\n-------------------------------------\n")